# Transformación de variables: encoding y escalado
**Minería de Datos I — ISTEA (Clase 5)** · Prof. Kevin Kovacs

**Objetivo:** convertir un dataset con variables de texto y escalas distintas en una tabla numérica lista para un modelo.
**Datos:** [Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance) (Kaggle).
**Qué hice:** usé LabelEncoder para `smoker` (dos valores), OneHotEncoder para `sex` y `region` (categorías sin orden) y MinMaxScaler para `age`, `bmi` y `charges`.
**Resultado principal:** un dataset de 1.191 filas con todas sus columnas numéricas y en la misma escala.
**Qué aprendí:** cada tipo de variable necesita una transformación distinta, y conviene tomar los nombres de las columnas del propio encoder en lugar de escribirlos a mano.

*Notebook desarrollado durante la cursada de Minería de Datos I (ISTEA).*

## Datos
Este notebook usa el archivo **`insurance.csv`**, del dataset [Medical Cost Personal Datasets (Kaggle)](https://www.kaggle.com/datasets/mirichoi0218/insurance).
Para ejecutarlo en Colab, descargá el archivo y subilo a la sesión (ícono de carpeta a la izquierda) antes de correr las celdas.

In [1]:
import pandas as pd

df = pd.read_csv('insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Paso previo: eliminamos los outliers con el criterio de Tukey (ver notebook 01).

In [2]:
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

In [3]:
for col in numeric_columns:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  sup = q3 + 1.5 * iqr
  inf = q1 - 1.5 * iqr
  outliers_indices = df[(df[col] > sup) | (df[col] < inf)].index
  df = df.drop(outliers_indices)

In [4]:
df_temp = df.copy()

# Transformaciones

Pasamos variables categoricas a numericas

## Ordinales

In [5]:
df.smoker.unique()

<StringArray>
['yes', 'no']
Length: 2, dtype: str

In [6]:
from sklearn.preprocessing import LabelEncoder

In [7]:
encoder = LabelEncoder()

In [8]:
df.smoker = encoder.fit_transform(df.smoker)

In [9]:
df.smoker.unique()

array([1, 0])

## Nominales

In [10]:
df.sex.unique()

<StringArray>
['female', 'male']
Length: 2, dtype: str

In [11]:
df.region.unique()

<StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str

In [12]:
categorical_columns = df.select_dtypes(include = 'object').columns

In [13]:
col_nominal = df[categorical_columns]

In [14]:
from sklearn.preprocessing import OneHotEncoder

In [15]:
enc = OneHotEncoder()

In [16]:
col_nominal = enc.fit_transform(col_nominal).toarray()

In [17]:
col_nominal

array([[1., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 1., 0.],
       [0., 1., 0., 0., 1., 0.],
       ...,
       [1., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0.]], shape=(1191, 6))

In [18]:
df = df.reset_index(drop = True)

Los nombres de las columnas nuevas se toman del propio encoder (`get_feature_names_out`). Así se respeta el orden en que crea las categorías, que es alfabético.

In [19]:
col_nominal = pd.DataFrame(col_nominal, columns = enc.get_feature_names_out())

In [20]:
col_nominal

,sex_female,sex_male,region_northeast,region_northwest,region_southeast,region_southwest
0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,1.0,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...
1186,0.0,1.0,0.0,1.0,0.0,0.0
1187,1.0,0.0,1.0,0.0,0.0,0.0
1188,1.0,0.0,0.0,0.0,1.0,0.0
1189,1.0,0.0,0.0,0.0,0.0,1.0


In [21]:
df.drop(categorical_columns, axis = 1, inplace = True)

In [22]:
df = pd.concat([df, col_nominal], axis = 1)

In [23]:
df

,age,bmi,children,smoker,charges,sex_female,sex_male,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,1,16884.92400,1.0,0.0,0.0,0.0,0.0,1.0
1,18,33.770,1,0,1725.55230,0.0,1.0,0.0,0.0,1.0,0.0
2,28,33.000,3,0,4449.46200,0.0,1.0,0.0,0.0,1.0,0.0
3,33,22.705,0,0,21984.47061,0.0,1.0,0.0,1.0,0.0,0.0
4,32,28.880,0,0,3866.85520,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1186,50,30.970,3,0,10600.54830,0.0,1.0,0.0,1.0,0.0,0.0
1187,18,31.920,0,0,2205.98080,1.0,0.0,1.0,0.0,0.0,0.0
1188,18,36.850,0,0,1629.83350,1.0,0.0,0.0,0.0,1.0,0.0
1189,21,25.800,0,0,2007.94500,1.0,0.0,0.0,0.0,0.0,1.0


## Variables numericas

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1191 entries, 0 to 1190
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               1191 non-null   int64  
 1   bmi               1191 non-null   float64
 2   children          1191 non-null   int64  
 3   smoker            1191 non-null   int64  
 4   charges           1191 non-null   float64
 5   sex_female        1191 non-null   float64
 6   sex_male          1191 non-null   float64
 7   region_northeast  1191 non-null   float64
 8   region_northwest  1191 non-null   float64
 9   region_southeast  1191 non-null   float64
 10  region_southwest  1191 non-null   float64
dtypes: float64(8), int64(3)
memory usage: 102.5 KB


In [25]:
col_num = ['age', 'bmi', 'charges']

In [26]:
data_num = df[col_num]

In [27]:
from sklearn.preprocessing import MinMaxScaler

instanciamos

In [28]:
scaler = MinMaxScaler()

In [29]:
data_num = scaler.fit_transform(data_num)

In [30]:
data_num

array([[0.02173913, 0.38778824, 0.47505834],
       [0.        , 0.57843456, 0.01819334],
       [0.2173913 , 0.55342644, 0.10028506],
       ...,
       [0.        , 0.67846703, 0.01530861],
       [0.06521739, 0.31958428, 0.02670394],
       [0.93478261, 0.42578759, 0.84443624]], shape=(1191, 3))

In [31]:
data_num = pd.DataFrame(data_num, columns = col_num)

In [32]:
data_num

,age,bmi,charges
0,0.021739,0.387788,0.475058
1,0.000000,0.578435,0.018193
2,0.217391,0.553426,0.100285
3,0.326087,0.219065,0.628746
4,0.304348,0.419617,0.082727
...,...,...,...
1186,0.695652,0.487496,0.285663
1187,0.000000,0.518350,0.032672
1188,0.000000,0.678467,0.015309
1189,0.065217,0.319584,0.026704


In [33]:
df.drop(col_num, axis = 1, inplace = True)

In [34]:
df = pd.concat([df, data_num], axis = 1)

# dataframe final

In [35]:
df_temp

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [36]:
df

,children,smoker,sex_female,sex_male,region_northeast,region_northwest,region_southeast,region_southwest,age,bmi,charges
0,0,1,1.0,0.0,0.0,0.0,0.0,1.0,0.021739,0.387788,0.475058
1,1,0,0.0,1.0,0.0,0.0,1.0,0.0,0.000000,0.578435,0.018193
2,3,0,0.0,1.0,0.0,0.0,1.0,0.0,0.217391,0.553426,0.100285
3,0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.326087,0.219065,0.628746
4,0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.304348,0.419617,0.082727
...,...,...,...,...,...,...,...,...,...,...,...
1186,3,0,0.0,1.0,0.0,1.0,0.0,0.0,0.695652,0.487496,0.285663
1187,0,0,1.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.518350,0.032672
1188,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.000000,0.678467,0.015309
1189,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.065217,0.319584,0.026704


## Conclusiones
- Una variable sin orden, como `region`, no se puede pasar a 0, 1, 2 y 3: el modelo interpretaría que hay un orden entre las regiones. Por eso se usa OneHotEncoder, que crea una columna por categoría.
- `smoker` tiene solo dos valores, así que alcanza con LabelEncoder (0 = no, 1 = sí).
- MinMaxScaler lleva `age`, `bmi` y `charges` al rango 0–1. Sin escalar, `charges` (miles de dólares) pesaría mucho más que `age` en modelos basados en distancias, como KMeans.
- En la versión de clase, los nombres de las columnas de región estaban escritos a mano y en un orden distinto del que usa el encoder (alfabético), por lo que quedaban mal etiquetadas. Usar `get_feature_names_out()` evita ese error.